In [ ]:
!python -m pip install pyserini

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.7/159.7 MB 6.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.3/413.3 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.4/197.4 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.3/96.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.7/67.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.2 MB/s eta 0

In [ ]:
!apt-get update
!apt-get install -y openjdk-21-jdk-headless

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,233 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [345 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,637 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

In [ ]:
!pip install pytrec_eval

  Preparing metadata (setup.py) ... done
  Created wheel for pytrec_eval: filename=pytrec_eval-0.5-cp312-cp312-linux_x86_64.whl size=309345 sha256=25e24f3922fcb3b8d547c992916ecd21d7d5a05032657f3d38835a226831eb9b
  Stored in directory: /root/.cache/pip/wheels/c6/4a/9e/e17f9ea004e1c221bd0ff384732285211c4917b790d598ea51
Successfully built pytrec_eval


In [ ]:
import os
import json
import gzip
import requests
from tqdm import tqdm

# 1. Setup Paths
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
output_file = "data/processed/docs.jsonl"

# 2. Define MIRACL Arabic File Sources
# MIRACL stores Arabic data in 5 parts (docs-0 to docs-4)
base_url = "https://huggingface.co/datasets/miracl/miracl-corpus/resolve/main/miracl-corpus-v1.0-ar/docs-{}.jsonl.gz"
num_files = 5

print(f"Downloading and Processing {num_files} chunks...")

with open(output_file, 'w', encoding='utf-8') as f_out:
    for i in range(num_files):
        file_url = base_url.format(i)
        temp_zip = f"data/raw/docs-{i}.jsonl.gz"

        # A. Download chunk
        print(f"Downloading chunk {i+1}/{num_files}...")
        response = requests.get(file_url, stream=True)
        with open(temp_zip, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024*1024):
                if chunk: f.write(chunk)

        # B. Read, Convert, and Append
        print(f"Processing chunk {i+1}...")
        with gzip.open(temp_zip, 'rt', encoding='utf-8') as f_in:
            for line in f_in:
                doc = json.loads(line)
                # Convert to Pyserini format (Concatenate Title + Text)
                obj = {
                    "id": doc['docid'],
                    "contents": f"{doc['title']} {doc['text']}"
                }
                f_out.write(json.dumps(obj, ensure_ascii=False) + '\n')

        # C. Cleanup to save disk space
        os.remove(temp_zip)

print(f"✅ Data successfully prepared at: {output_file}")

Processing chunk 1...
Processing chunk 2...
Processing chunk 3...
Processing chunk 4...
Processing chunk 5...
✅ Data successfully prepared at: data/processed/docs.jsonl


In [ ]:
# Clear old attempts
!rm -rf indexes/my_local_index

# Build Index (~15 mins)
!python -m pyserini.index.lucene \
  --collection JsonCollection \
  --input data/processed \
  --index indexes/my_local_index \
  --generator DefaultLuceneDocumentGenerator \
  --threads 12 \
  --storePositions --storeDocvectors --storeRaw \
  --language arabic

2026-01-10 12:09:28,112 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:208) - Setting log level to INFO
2026-01-10 12:09:28,119 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:211) - ============ Loading Index Configuration ============
2026-01-10 12:09:28,119 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:212) - AbstractIndexer settings:
2026-01-10 12:09:28,141 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:213) -  + DocumentCollection path: data/processed
2026-01-10 12:09:28,142 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:214) -  + CollectionClass: JsonCollection
2026-01-10 12:09:28,143 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:215) -  + Index path: indexes/my_local_index
2026-01-10 12:09:28,144 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:216) -  + Threads: 12
2026-01-10 12:09:28,145 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:217) -  + Optimize (merge segments)? false
Jan 10, 2026 12:09:28 P

In [ ]:
from pyserini.search.lucene import LuceneSearcher
from pyserini.search import get_topics, get_qrels
import pytrec_eval
from tqdm import tqdm

# 1. Load Index
print("Loading Local Index...")
searcher = LuceneSearcher('indexes/my_local_index')
searcher.set_language('arabic')

# --- TUNING TO MATCH OFFICIAL BASELINE ---
# Official Anserini often uses these for MS MARCO / MIRACL defaults
# If this doesn't match perfectly, we stick with 1.2/0.75, but let's try.
searcher.set_bm25(k1=0.9, b=0.4)
# -----------------------------------------

# 2. Retrieval (k=1000 to match Recall@100 metric)
topics = get_topics('miracl-v1.0-ar-dev')
output_file = "results/baseline/official_config_run.txt"

print(f"Retrieving Top 1000 for {len(topics)} queries...")
with open(output_file, 'w') as f:
    for qid in tqdm(topics.keys()):
        query_text = topics[qid]['title']
        # MATCHING SCREENSHOT: --hits 1000
        hits = searcher.search(query_text, k=1000)

        for i, hit in enumerate(hits):
            f.write(f"{qid} Q0 {hit.docid} {i+1} {hit.score:.5f} tuned_bm25\n")

# 3. Evaluation
print("Evaluating...")
raw_qrels = get_qrels('miracl-v1.0-ar-dev')
qrels = {str(q): {str(d): int(s) for d, s in v.items()} for q, v in raw_qrels.items()}

with open(output_file, 'r') as f:
    run_data = pytrec_eval.parse_run(f)

# We measure BOTH Recall@100 (for official comparison) and Recall@10 (for your thesis)
evaluator = pytrec_eval.RelevanceEvaluator(
    qrels, {'recall_10', 'recall_100', 'ndcg_cut_10', 'recip_rank'}
)
results = evaluator.evaluate(run_data)

# Aggregate
metrics = ['recall_10', 'recall_100', 'ndcg_cut_10', 'recip_rank']
aggs = {m: 0.0 for m in metrics}

for qid in results:
    for m in metrics:
        aggs[m] += results[qid][m]

print("\n" + "="*40)
print("🎯 FINAL COMPARISON RESULTS")
print("="*40)
print(f"Recall@100: {aggs['recall_100']/len(results):.4f}  <-- Should be ~0.889")
print(f"NDCG@10:    {aggs['ndcg_cut_10']/len(results):.4f}  <-- Should be ~0.481")
print("-" * 40)
print(f"Recall@10:  {aggs['recall_10']/len(results):.4f}  (Your Thesis Metric)")
print("="*40)

Loading Local Index...
Retrieving Top 1000 for 2896 queries...


100%|██████████| 2896/2896 [03:15<00:00, 14.85it/s]


Evaluating...

🎯 FINAL COMPARISON RESULTS
Recall@100: 0.7860  <-- Should be ~0.889
NDCG@10:    0.4320  <-- Should be ~0.481
----------------------------------------
Recall@10:  0.5485  (Your Thesis Metric)
